In [15]:
#StandardScaler         → feature scaling (zero-mean, unit-variance)
#joblib                 → serialize the scaler to disk

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import os
import warnings

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────
# CONFIGURATION — all magic numbers in one place.
# Changing a value here changes the entire pipeline.
# ─────────────────────────────────────────────────────

RAW_PATH       = '../data/raw/diabetes.csv'
CLEAN_PATH     = '../data/processed/diabetes_clean.csv'
SCALER_PATH    = '../artifacts/scalers/standard_scaler.pkl'

# Columns that use 0 as a stand-in for missing (from EDA)
ZERO_AS_MISSING = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Train-test split parameters
TEST_SIZE      = 0.20       # 20% held out for final evaluation
RANDOM_STATE   = 42         # fixed seed → reproducible splits every time

# Ensure output directories exist
os.makedirs('data/processed', exist_ok=True)
os.makedirs('artifacts/scalers', exist_ok=True)

print('✅ Configuration loaded')

✅ Configuration loaded


In [16]:
df = pd.read_csv(RAW_PATH)

print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

Loaded: 768 rows × 9 columns


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [17]:
# ─────────────────────────────────────────────────────
# WHY replace with NaN and not just impute directly?
# • Makes the missing values VISIBLE to pandas (isnull())
# • Allows us to verify counts before and after imputation
# • Any future imputation method (KNN, MICE, etc.) works
#   on NaN by convention
#
# We use .replace({0: np.nan}) ONLY on the 5 target columns.
# Pregnancies = 0 is VALID and must NOT be touched.
# ─────────────────────────────────────────────────────

# Snapshot counts before replacement
before = df[ZERO_AS_MISSING].eq(0).sum()

# Replace
df[ZERO_AS_MISSING] = df[ZERO_AS_MISSING].replace(0, np.nan)

# Verify
after = df[ZERO_AS_MISSING].isna().sum()

check = pd.DataFrame({'Before (zeros)': before, 'After (NaNs)': after})
print(check)
print(f"\nTotal NaN cells now: {df.isna().sum().sum()}")

               Before (zeros)  After (NaNs)
Glucose                     5             5
BloodPressure              35            35
SkinThickness             227           227
Insulin                   374           374
BMI                        11            11

Total NaN cells now: 652


In [18]:
# ─────────────────────────────────────────────────────
# WHY median and not mean?
#
# Insulin is extremely right-skewed:
#   mean = 79.8  vs  median = 30.5
#
# If we impute with the mean, we inject a value (79.8)
# that does NOT represent the typical patient.
# Median (30.5) is the true "middle" patient and is
# unaffected by the extreme tail.
#
# The same logic applies to all 5 columns: their
# distributions are skewed, so median is safer.
# ─────────────────────────────────────────────────────

# Capture medians BEFORE imputing (for transparency)
medians = df[ZERO_AS_MISSING].median()
print('Imputation values (medians):')
print(medians)
print()

# Fill NaN with median
df[ZERO_AS_MISSING] = df[ZERO_AS_MISSING].fillna(medians)

# Verify: zero NaN remaining
remaining_nan = df.isna().sum().sum()
print(f'NaN remaining after imputation: {remaining_nan}')
assert remaining_nan == 0, '❌ Imputation incomplete!'
print('✅ All missing values filled')

Imputation values (medians):
Glucose          117.0
BloodPressure     72.0
SkinThickness     29.0
Insulin          125.0
BMI               32.3
dtype: float64

NaN remaining after imputation: 0
✅ All missing values filled


In [19]:
# ─────────────────────────────────────────────────────
# WHY engineer new features?
#
# The original 8 features capture individual measurements.
# But diabetes risk is driven by INTERACTIONS between them.
# A 25-year-old with high glucose is very different from a
# 60-year-old with the same glucose.
#
# NEW FEATURES:
#
# 1. Age_Glucose_Interaction
#    = Age × Glucose
#    WHY: Older patients with elevated glucose carry
#         compounded risk. This term lets tree models
#         split on the combined signal directly.
#
# 2. BMI_Age_Interaction
#    = BMI × Age
#    WHY: High BMI matters more at older ages (metabolic
#         decline). This captures that non-linearity.
#
# 3. Glucose_BMI_Interaction
#    = Glucose × BMI
#    WHY: Both are the two strongest predictors (corr 0.467
#         and 0.293). Their product amplifies the signal.
# ─────────────────────────────────────────────────────

df['Age_Glucose_Interaction']  = df['Age'] * df['Glucose']
df['BMI_Age_Interaction']      = df['BMI'] * df['Age']
df['Glucose_BMI_Interaction']  = df['Glucose'] * df['BMI']

print(f'New shape after feature engineering: {df.shape}')
print(f'New columns added: Age_Glucose_Interaction, BMI_Age_Interaction, Glucose_BMI_Interaction')
df[['Age_Glucose_Interaction', 'BMI_Age_Interaction', 'Glucose_BMI_Interaction']].describe()

New shape after feature engineering: (768, 12)
New columns added: Age_Glucose_Interaction, BMI_Age_Interaction, Glucose_BMI_Interaction


,Age_Glucose_Interaction,BMI_Age_Interaction,Glucose_BMI_Interaction
count,768.000000,768.000000,768.000000
mean,4139.380208,1080.906771,3996.667188
std,2080.608356,437.813986,1469.660470
min,1232.000000,382.200000,1100.000000
25%,2608.500000,744.800000,2924.100000
50%,3480.000000,987.250000,3750.150000
75%,5130.750000,1357.200000,4800.250000
max,12998.000000,2697.000000,10692.000000


In [20]:
# ─────────────────────────────────────────────────────
# WHY cap and not remove?
# • We only have 768 rows. Removing outlier rows
#   shrinks the training set and can bias the model
#   against the very patients who most need screening.
# • Capping PRESERVES the row count while neutralising
#   extreme values that could dominate gradient-based
#   or distance-based models.
#
# HOW winsorization works:
#   Any value below lower_fence  → set to lower_fence
#   Any value above upper_fence  → set to upper_fence
#   Everything in between        → untouched
#
# We apply this ONLY to the original 8 features.
# The engineered interaction features will naturally
# fall into range once their inputs are capped.
# ─────────────────────────────────────────────────────

ORIGINAL_FEATURES = ['Pregnancies', 'Glucose', 'BloodPressure',
                     'SkinThickness', 'Insulin', 'BMI',
                     'DiabetesPedigreeFunction', 'Age']

cap_log = []  # record what was changed for transparency

for col in ORIGINAL_FEATURES:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    # Count before capping
    n_below = (df[col] < lower).sum()
    n_above = (df[col] > upper).sum()
    
    # Cap
    df[col] = df[col].clip(lower=lower, upper=upper)
    
    cap_log.append({
        'Feature': col,
        'Lower Cap': round(lower, 3),
        'Upper Cap': round(upper, 3),
        'Capped Below': n_below,
        'Capped Above': n_above
    })

cap_df = pd.DataFrame(cap_log)
print(cap_df.to_string(index=False))
print(f"\n✅ Outlier capping complete — {sum(cap_df['Capped Below'] + cap_df['Capped Above'])} values adjusted")

                 Feature  Lower Cap  Upper Cap  Capped Below  Capped Above
             Pregnancies     -6.500     13.500             0             4
                 Glucose     39.000    201.000             0             0
           BloodPressure     40.000    104.000             4            10
           SkinThickness     14.500     42.500            39            48
                 Insulin    112.875    135.875           173           173
                     BMI     13.850     50.250             0             8
DiabetesPedigreeFunction     -0.330      1.200             0            29
                     Age     -1.500     66.500             0             9

✅ Outlier capping complete — 497 values adjusted


In [21]:
# ─────────────────────────────────────────────────────
# Separate X (features) from y (target) BEFORE scaling.
# The target (0/1) must never be scaled — it's categorical.
# ─────────────────────────────────────────────────────

X = df.drop(columns=['Outcome'])
y = df['Outcome']

print(f'X shape: {X.shape}   →  {X.shape[1]} features')
print(f'y shape: {y.shape}   →  target distribution preserved:')
print(y.value_counts())

X shape: (768, 11)   →  11 features
y shape: (768,)   →  target distribution preserved:
Outcome
0    500
1    268
Name: count, dtype: int64


In [22]:
# ─────────────────────────────────────────────────────
# WHY stratified?
#
# The target is imbalanced (65.1% vs 34.9%).
# A random split could accidentally put almost all
# diabetic cases in train and leave test with too few.
#
# stratify=y forces BOTH sets to maintain the same
# 65/35 class ratio as the original dataset.
#
# WHY 80/20?
# • 80% train gives enough data to learn patterns
# • 20% test (≈154 rows) is enough for reliable metrics
# • Common industry-standard split for datasets < 1000 rows
# ─────────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y               # ← critical: preserves class balance
)

print(f'Train set: {X_train.shape[0]} rows   →  Diabetic: {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Test set:  {X_test.shape[0]} rows   →  Diabetic: {y_test.sum()} ({y_test.mean()*100:.1f}%)')
print(f'\n✅ Class ratio preserved in both sets (≈35% diabetic)')

Train set: 614 rows   →  Diabetic: 214 (34.9%)
Test set:  154 rows   →  Diabetic: 54 (35.1%)

✅ Class ratio preserved in both sets (≈35% diabetic)


In [23]:
# ─────────────────────────────────────────────────────
# WHY StandardScaler?
# Transforms each feature to: (value - mean) / std
# → Every feature ends up with mean=0 and std=1
#
# This matters because:
# • KNN uses Euclidean distance → features with larger
#   ranges (e.g. Insulin: 0–846) would dominate distance
#   calculations over small-range features (e.g. DPF: 0–2.4)
# • Logistic Regression & SVM converge faster when features
#   are on the same scale
# • Random Forest does NOT need scaling (it splits on
#   thresholds) — but we scale anyway for a fair comparison
#   across all models in Notebooks 03 and 04
#
# ⚠️  CRITICAL RULE — FIT ONLY ON TRAIN, TRANSFORM BOTH:
# • fit_transform(X_train) → learn mean & std from training data
# • transform(X_test)      → apply SAME mean & std to test data
# • If we fit on combined data, test set information "leaks"
#   into training → inflated metrics → useless model
# ─────────────────────────────────────────────────────

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # fit + transform train
X_test_scaled  = scaler.transform(X_test)        # transform test only

# Convert back to DataFrames (keeps column names for readability)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns, index=X_test.index)

print('Scaled Train — first 5 rows:')
print(X_train_scaled.head())
print(f'\nMean per feature (should be ≈0): {X_train_scaled.mean().round(3).tolist()}')
print(f'Std  per feature (should be ≈1): {X_train_scaled.std().round(3).tolist()}')

Scaled Train — first 5 rows:
     Pregnancies   Glucose  BloodPressure  SkinThickness   Insulin       BMI  \
353    -0.855120 -1.056427      -0.858066      -1.910489 -1.475845 -0.784098   
711     0.360963  0.144399       0.500239      -0.243007 -1.475845 -0.421343   
373    -0.551100 -0.556083      -1.197642       1.491175 -1.475845  0.379741   
46     -0.855120  0.811525      -1.367431       0.023790  0.039775 -0.406228   
682    -1.159141 -0.889646      -0.688278       1.357776 -1.475845  1.845875   

     DiabetesPedigreeFunction       Age  Age_Glucose_Interaction  \
353                  0.400579 -0.798419                -0.955438   
711                 -0.090600  0.572372                 0.418753   
373                 -0.836078 -0.712745                -0.733563   
46                   0.344842 -0.370047                 0.034171   
682                 -0.344899 -0.969768                -0.988838   

     BMI_Age_Interaction  Glucose_BMI_Interaction  
353            -0.981042     

In [24]:
# ─────────────────────────────────────────────────────
# SAVE 1 — Cleaned (pre-scale) CSV
# This is the human-readable version for inspection.
# It has imputed values, capped outliers, and engineered
# features, but is NOT scaled (numbers still in original units).
# ─────────────────────────────────────────────────────

clean_df = pd.concat([X, y], axis=1)
clean_df.to_csv(CLEAN_PATH, index=False)
print(f'✅ Saved cleaned CSV → {CLEAN_PATH}')

# ─────────────────────────────────────────────────────
# SAVE 2 — Fitted Scaler
# joblib serialises the scaler object (with its learned
# mean and std values) to a .pkl file.
# The backend API and model notebooks load this to apply
# the EXACT same transformation to new patient inputs.
# ─────────────────────────────────────────────────────

joblib.dump(scaler, SCALER_PATH)
print(f'✅ Saved scaler      → {SCALER_PATH}')

✅ Saved cleaned CSV → ../data/processed/diabetes_clean.csv
✅ Saved scaler      → ../artifacts/scalers/standard_scaler.pkl


In [26]:
# ─────────────────────────────────────────────────────
# FINAL VERIFICATION — print everything the next notebooks need
# ─────────────────────────────────────────────────────

summary = """
╔═══════════════════════════════════════════════════════╗
║          PREPROCESSING PIPELINE COMPLETE              ║
╠═══════════════════════════════════════════════════════╣
║                                                       ║
║  Steps executed:                                      ║
║    [1] Zeros → NaN       (5 columns)                  ║
║    [2] Median imputation (all NaN filled)             ║
║    [3] Feature engineering (3 new interaction terms)  ║
║    [4] Outlier capping   (IQR winsorization)          ║
║    [5] Standard scaling  (fit on train only)          ║
║    [6] Stratified 80/20 split                         ║
║                                                       ║
║  Outputs ready for Notebooks 03 & 04:                 ║
║    X_train_scaled  shape → {x_tr}                     ║
║    X_test_scaled   shape → {x_te}                     ║
║    y_train         shape → {y_tr}                     ║
║    y_test          shape → {y_te}                     ║
║    Features used         → {n_feat} total             ║
║                                                       ║
╚═══════════════════════════════════════════════════════╝
""".format(
    x_tr=X_train_scaled.shape,
    x_te=X_test_scaled.shape,
    y_tr=y_train.shape,
    y_te=y_test.shape,
    n_feat=X_train_scaled.shape[1]
)

print(summary)




╔═══════════════════════════════════════════════════════╗
║          PREPROCESSING PIPELINE COMPLETE              ║
╠═══════════════════════════════════════════════════════╣
║                                                       ║
║  Steps executed:                                      ║
║    [1] Zeros → NaN       (5 columns)                  ║
║    [2] Median imputation (all NaN filled)             ║
║    [3] Feature engineering (3 new interaction terms)  ║
║    [4] Outlier capping   (IQR winsorization)          ║
║    [5] Standard scaling  (fit on train only)          ║
║    [6] Stratified 80/20 split                         ║
║                                                       ║
║  Outputs ready for Notebooks 03 & 04:                 ║
║    X_train_scaled  shape → (614, 11)                     ║
║    X_test_scaled   shape → (154, 11)                     ║
║    y_train         shape → (614,)                     ║
║    y_test          shape → (154,)                     ║
║    Fe